In [ ]:
import sys
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

sys.path.append('../')
from data_provider.data_loader import load_dataset

#### Load dataset

In [16]:
path_to_folder = '../data'
filename = 'AI_Detection' # Change to dataset you wish to train on

df = load_dataset(path_to_folder, filename)

# Check the column names to ensure features and labels are loaded correctly
print(df.head())
X = df['text'].tolist()
y = df['label'].tolist()

                                                text  label
0  Phones\n\nModern humans today are always on th...      0
1  This essay will explain if drivers should or s...      0
2  Driving while the use of cellular devices\n\nT...      0
3  Phones & Driving\n\nDrivers should not be able...      0
4  Cell Phone Operation While Driving\n\nThe abil...      0


#### Split dataset into training and test sets

In [17]:
# Dataset is not randomly ordered, so we will shuffle it to ensure a representative split between training and testing sets

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2026,
    stratify=y,
    shuffle=True
)

#### Vectorize text data using TF-IDF

In [18]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

#### Logistic Regression Model

In [19]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return self.linear(x)

#### Training Loop for Logistic Regression Model

In [29]:
def run_logistic_regression(X_train, y_train, X_test, y_test):
    model = LogisticRegressionModel(X_train.shape[1])
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

    num_epochs = 10
    batch_size = 32

    for epoch in range(num_epochs):
        model.train()
        for i in tqdm(range(0, len(X_train_tensor), batch_size)):
            X_batch = X_train_tensor[i:i+batch_size]
            y_batch = y_train_tensor[i:i+batch_size]

            logits = model.forward(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
        logits = model.forward(X_test_tensor)
        probabilities = torch.sigmoid(logits)
        predicted_labels = (probabilities >= 0.5).float()

        correct = (predicted_labels == y_test_tensor).sum()
        accuracy = correct / len(y_test_tensor)
        print(f'Test Accuracy: {accuracy.item()}')

        y_true = y_test_tensor.flatten().numpy()
        y_pred = predicted_labels.flatten().numpy()
        print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))



#### Train our Logistic Regression model

In [30]:
run_logistic_regression(X_train, y_train, X_test, y_test)

100%|██████████| 2152/2152 [00:00<00:00, 3979.36it/s]


Epoch 1/10, Loss: 0.3597334027290344


100%|██████████| 2152/2152 [00:00<00:00, 4042.48it/s]


Epoch 2/10, Loss: 0.28714489936828613


100%|██████████| 2152/2152 [00:00<00:00, 4136.63it/s]


...

Epoch 8/10, Loss: 0.08917537331581116


100%|██████████| 2152/2152 [00:00<00:00, 4147.30it/s]


Epoch 9/10, Loss: 0.07491040229797363


100%|██████████| 2152/2152 [00:00<00:00, 3718.60it/s]

Epoch 10/10, Loss: 0.06347901374101639
Test Accuracy: 0.9893672466278076
              precision    recall  f1-score   support

       Human       0.98      0.99      0.99      7700
          AI       0.99      0.99      0.99      9511

    accuracy                           0.99     17211
   macro avg       0.99      0.99      0.99     17211
weighted avg       0.99      0.99      0.99     17211

